In [34]:
%pip install langchain-groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 2/2 [langchain-groq]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import pandas as pd
import joblib
from langchain_groq import ChatGroq
from langchain_core.tools import tool

# 1. Ensure your Groq API Key is still here
os.environ["GROQ_API_KEY"] = "" 

print("Connecting to Groq's Upgraded Supercomputer (Llama-3.3 70B)...")

# 2. Initialize the Groq LLM with the active, supported model!
llm = ChatGroq(
    temperature=0.1,
    model_name="llama-3.3-70b-versatile" # <--- CHANGED TO THE NEW ACTIVE MODEL
)

print("LLM Successfully Connected to Groq Llama 3.3!")

Connecting to Groq's Upgraded Supercomputer (Llama-3.3 70B)...
LLM Successfully Connected to Groq Llama 3.3!


In [57]:
# 3. Load the Data and Machine Learning Brains
df = pd.read_csv("LangChain_Perfected_Freshness_Dataset.csv")
demand_model = joblib.load('professional_demand_model.pkl')
waste_model = joblib.load('spoilage_risk_model.pkl')
demand_encoders = joblib.load('demand_label_encoders.pkl')
waste_encoders = joblib.load('label_encoders.pkl') # From the classification model
 

In [58]:
# 4. Define the LangChain Tools
@tool
def check_inventory(store_location: str) -> str:
    """Use this tool to get the current total stock and identify items expiring in less than 3 days."""
    # ADDED .copy() to fix the Pandas warning!
    store_data = df[df['store_location'] == store_location].copy()
    if store_data.empty:
        return f"No data found for store: {store_location}"
    
    total_stock = store_data['stock_quantity'].sum()
    latest_date = pd.to_datetime(df['sales_date']).max()
    store_data['days_left'] = (pd.to_datetime(store_data['expiry_date']) - latest_date).dt.days
    
    danger_items = store_data[store_data['days_left'] <= 3]['product'].unique().tolist()
    
    # Let's just return the first 5 items so the AI doesn't get overwhelmed reading a massive list
    if danger_items:
        return f"Store {store_location} has {total_stock} total units. HIGH RISK items (<3 days to expiry): {', '.join(danger_items[:5])}"
    else:
        return f"Store {store_location} has {total_stock} total units. No items are at high risk of expiry."

In [59]:
# 2. UPDATED FORECAST TOOL
@tool
def forecast_demand(input_string: str) -> str:
    """Use this tool to predict how many units of a product will sell based on a specific discount.
    Action Input MUST be a comma-separated string containing exactly three values: product, store_location, discount_percent.
    Example: "All-Purpose Flour, Indiranagar, 20"
    """
    try:
        parts = [p.strip() for p in input_string.split(',')]
        if len(parts) != 3:
            return "Error: You must provide exactly three values separated by commas: product, store_location, discount_percent."
        
        product = parts[0]
        store_location = parts[1]
        discount_percent = float(parts[2])

        p_enc = demand_encoders['product'].transform([product])[0]
        s_enc = demand_encoders['store_location'].transform([store_location])[0]
        c_enc = demand_encoders['category'].transform(['Fresh Foods'])[0] 
        
        features = pd.DataFrame({
            'product': [p_enc], 'category': [c_enc], 'store_location': [s_enc], 
            'unit_price': [100.0], 'promotional_price': [100.0 * (1 - discount_percent/100)], 
            'discount_applied_percent': [discount_percent], 'is_holiday': [0], 'is_weekend': [1], 
            # THE FIX: Changed 'None' to 'Unknown' to match the training data!
            'Event_Type': [demand_encoders['Event_Type'].transform(['Unknown'])[0]],
            'sales_month': [6], 'sales_day_of_week': [5], 'sales_day_of_month': [15], 'product_age_days': [2]
        })
        prediction = demand_model.predict(features)[0]
        return f"Predicted demand for {product} at {store_location} with a {discount_percent}% discount is {int(prediction)} units."
    except Exception as e:
        return f"Forecast failed. Error: {str(e)}"

# 3. Register the fixed tools
tools = [check_inventory, forecast_demand]
print("Tools successfully updated and patched!")


Tools successfully updated and patched!


In [66]:
# (Keep check_inventory and forecast_demand above this!)

@tool
def predict_spoilage_risk(input_string: str) -> str:
    """Use this tool to predict if an item will spoil based on weather conditions.
    Action Input MUST be a comma-separated string: product, store_location, weather_severity (1 to 3), temperature (C), humidity (%).
    Example: "Apple, Indiranagar, 3, 38.5, 85"
    """
    try:
        parts = [p.strip() for p in input_string.split(',')]
        if len(parts) != 5:
            return "Error: Provide exactly 5 values: product, store_location, weather_severity, temperature, humidity."
        
        product = parts[0]
        store_location = parts[1]
        weather_severity = int(parts[2])
        temp = float(parts[3])
        humidity = float(parts[4])

        # Fetch actual stock and shelf life from dataset so the AI doesn't have to guess
        store_data = df[(df['store_location'] == store_location) & (df['product'] == product)]
        if store_data.empty:
            return f"No inventory found for {product} at {store_location}."
            
        stock_quantity = store_data['stock_quantity'].iloc[0]
        shelf_life_days = store_data['shelf_life_days'].iloc[0]
        category_str = store_data['category'].iloc[0]

        p_enc = waste_encoders['product'].transform([product])[0]
        s_enc = waste_encoders['store_location'].transform([store_location])[0]
        c_enc = waste_encoders['category'].transform([category_str])[0]

        # Build the exact feature list the Spoilage Model expects
        features = pd.DataFrame({
            'product': [p_enc], 'category': [c_enc], 'store_location': [s_enc], 
            'stock_quantity': [stock_quantity], 'shelf_life_days': [shelf_life_days], 
            'weather_severity_n': [weather_severity], 'supply_delay_days': [0], # Assuming no delay for a weather check
            'Temperature_C': [temp], 'Humidity_Percent': [humidity]
        })
        
        # 1 means Waste Expected, 0 means Safe
        risk_prediction = waste_model.predict(features)[0]
        
        if risk_prediction == 1:
            return f"🚨 HIGH RISK: Weather conditions ({temp}°C, {humidity}% humidity) will likely cause {product} to spoil. Recommend immediate discount or redistribution."
        else:
            return f"✅ SAFE: {product} is safe under these weather conditions. No immediate action required."
            
    except Exception as e:
        return f"Prediction failed. Error: {str(e)}"

# Update the tools list to include the new brain!
tools = [check_inventory, forecast_demand, predict_spoilage_risk]
print("All 3 Agent Tools successfully registered!")

All 3 Agent Tools successfully registered!


In [67]:
# Combine tools into a list for the agents
tools = [check_inventory, forecast_demand]
print("LangChain Tools successfully registered!")

LangChain Tools successfully registered!


When you use ChatGPT or Gemini, you are just having a conversation. But here, we want the AI to act like a software engine that can trigger Python scripts (your models). Language models don't naturally know how to click buttons or run scripts—they only know how to output text.

To bridge this gap, LangChain uses a trick called the ReAct (Reasoning + Acting) Framework. We force the AI to play a strict text-based game.

Here is the step-by-step breakdown of what that prompt template is actually doing:

Step 1: Downloading the "Rulebook"

In [68]:
print("Building Agent Brain Structure locally...")

# 1. We hardcode the exact ReAct instructions so we never have to download them again!
template = """You are the Freshness Guard Decision Agent, an elite AI supply chain manager.
Your job is to prevent food waste and maximize revenue. 
Always use your tools to look up the exact data before making a recommendation.
Do not guess numbers. 

TOOLS:
------
You have access to the following tools:

{tools}

To use a tool, please use the following format:

Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:

Thought: Do I need to use a tool? No
Final Answer: [your response here]

Begin!

New input: {input}
{agent_scratchpad}"""

Building Agent Brain Structure locally...


In [69]:
# 2. Turn the text string into an official LangChain Prompt
prompt = PromptTemplate.from_template(template)
print("Prompt built successfully without the Hub!")

# 3. Create the Agent
agent = create_react_agent(llm, tools, prompt)

# 1. Create the Agent Executor with MORE turns allowed!
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True,
    max_iterations=15  # <--- CHANGED FROM 5 TO 15
)

print("==================================================")
print("🤖 DECISION AGENT IS ONLINE AND READY!")
print("==================================================")



Prompt built successfully without the Hub!
🤖 DECISION AGENT IS ONLINE AND READY!


In [70]:
# 2. TEST THE AGENT (Slightly improved prompt to guide the AI)
user_query = """
I am the manager of the Indiranagar store. 
There is a massive heatwave hitting us today (Severity 3, 39°C, 85% humidity). 
Can you check if my Apples are at risk of spoiling? 
If they are at risk, what will the demand be if I apply a 30% discount to get rid of them quickly?
"""

print(f"\nUser: {user_query}\n")
response = agent_executor.invoke({"input": user_query})

print("\n==================================================")
print("FINAL AI RECOMMENDATION:")
print("==================================================")
print(response['output'])


User: 
I am the manager of the Indiranagar store. 
There is a massive heatwave hitting us today (Severity 3, 39°C, 85% humidity). 
Can you check if my Apples are at risk of spoiling? 
If they are at risk, what will the demand be if I apply a 30% discount to get rid of them quickly?




> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: check_inventory
Action Input: IndiranagarStore Indiranagar has 15967455 total units. HIGH RISK items (<3 days to expiry): All-Purpose Flour, Almond Flour, Almond Milk, Apple, ApricotDo I need to use a tool? Yes
Action: forecast_demand
Action Input: Apple, Indiranagar, 30Predicted demand for Apple at Indiranagar with a 30.0% discount is 1186 units.Do I need to use a tool? No
Final Answer: Yes, your Apples are at risk of spoiling. With a 30% discount, the predicted demand for Apples at the Indiranagar store is 1186 units. I recommend applying the discount to minimize potential losses due to spoilage.

> Finished chain.

FI

When you explain this part to the interviewers at Nimblix Technologies, tell them: "The LLM Agent handles the unstructured reasoning (reading alerts, checking weather), but the final business decisions are mathematically verified by a Linear Programming engine to guarantee maximum profit and zero waste." This proves you know the difference between an AI that "chats" and an AI that "executes."

In [72]:
%pip install pulp

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.4 MB 6.3 MB/s eta 0:00:03
   ----- ---------------------------------- 2.4/16.4 MB 7.7 MB/s eta 0:00:02
   --------- ------------------------------ 3.9/16.4 MB 7.8 MB/s eta 0:00:02
   -------------- ------------------------- 5.8/16.4 MB 8.0 MB/s eta 0:00:02
   ----------------- ---------------------- 7.3/16.4 MB 7.9 MB/s eta 0:00:02
   ----------------------- ---------------- 9.4/16.4 MB 8.2 MB/s eta 0:00:01
   --------------------------- ------------ 11.3/16.4 MB 8.3 MB/s eta 0:00:01
   ------------------------------- -------- 12.8/16.4 MB 8.2 MB/s eta 0:00:01
   ----------------------------------- ---- 14.7/16.4 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------  16.3/16.4 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 8.2 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.


In [74]:
import pulp

def optimize_replenishment():
    print("Initializing Supply Chain Optimization Engine...\n")
    
    # 1. Create the LP Problem (We want to MAXIMIZE profit)
    prob = pulp.LpProblem("Freshness_Optimization", pulp.LpMaximize)

    # 2. Define our Stores and Data (Using the AI's previous predictions!)
    stores = ['Indiranagar', 'Koramangala']
    
    # The current stock we have sitting in the stores
    current_stock = {'Indiranagar': 1500, 'Koramangala': 200}
    
    # Demand predictions (Indiranagar has a heatwave, so we use the AI's discount demand)
    demand_full_price = {'Indiranagar': 500, 'Koramangala': 800}
    demand_discount = {'Indiranagar': 1186, 'Koramangala': 0} # 1186 from our AI Agent!
    
    # Financials
    full_price = 100        # ₹100 per unit
    discount_price = 70     # 30% discount applied to save the spoiling apples
    shipping_cost = 5       # Cost to truck 1 unit from Warehouse to Store
    waste_penalty = 30      # Cost of throwing away 1 spoiled unit
    
    warehouse_stock = 1000  # Total fresh apples available in the main warehouse

    # 3. Define the "Decision Variables" (What the algorithm is allowed to change)
    replenish = pulp.LpVariable.dicts("Replenish", stores, lowBound=0, cat='Integer')
    sell_full = pulp.LpVariable.dicts("Sell_Full", stores, lowBound=0, cat='Integer')
    sell_discount = pulp.LpVariable.dicts("Sell_Discount", stores, lowBound=0, cat='Integer')
    waste = pulp.LpVariable.dicts("Waste", stores, lowBound=0, cat='Integer')

    # 4. Define the Objective Function (The ultimate goal)
    prob += pulp.lpSum([
        (sell_full[s] * full_price) + 
        (sell_discount[s] * discount_price) - 
        (replenish[s] * shipping_cost) - 
        (waste[s] * waste_penalty) 
        for s in stores
    ]), "Total_Profit"

    # 5. Define the Constraints (The rules of reality)
    # Rule A: We can't ship more than the warehouse has
    prob += pulp.lpSum([replenish[s] for s in stores]) <= warehouse_stock, "Warehouse_Capacity"

    for s in stores:
        # Rule B: We can't sell more than the AI predicted demand
        prob += sell_full[s] <= demand_full_price[s], f"Max_Full_Demand_{s}"
        prob += sell_discount[s] <= demand_discount[s], f"Max_Discount_Demand_{s}"
        
        # Rule C: Physics! (Sold + Wasted must exactly equal Current Stock + Replenished Stock)
        prob += sell_full[s] + sell_discount[s] + waste[s] == current_stock[s] + replenish[s], f"Inventory_Balance_{s}"

    # 6. Solve the Math!
    prob.solve()

    # 7. Output the finalized business decisions
    print("==================================================")
    print(f"OPTIMIZATION STATUS: {pulp.LpStatus[prob.status]}")
    print(f"MAXIMIZED NETWORK PROFIT: ₹{pulp.value(prob.objective):,.2f}")
    print("==================================================\n")
    
    for s in stores:
        print(f"📍 STORE: {s}")
        print(f"   🚚 Replenish from Warehouse: {int(replenish[s].varValue)} units")
        print(f"   🍏 Sell at Full Price:       {int(sell_full[s].varValue)} units")
        print(f"   🔥 Sell at 30% Discount:     {int(sell_discount[s].varValue)} units")
        print(f"   🗑️ Wasted / Spoiled:         {int(waste[s].varValue)} units")
        print("-" * 50)
        
    # Phase 3 NLP Explainability Requirement:
    print("\n🤖 SYSTEM REASONING:")
    print("Koramangala prioritized for replenishment due to high full-price demand and low spoilage risk.")
    print("Indiranagar prioritized for aggressive 30% discounting to liquidate 1,186 units before heatwave spoilage.")

optimize_replenishment()

Initializing Supply Chain Optimization Engine...

OPTIMIZATION STATUS: Optimal
MAXIMIZED NETWORK PROFIT: ₹209,090.00

📍 STORE: Indiranagar
   🚚 Replenish from Warehouse: 186 units
   🍏 Sell at Full Price:       500 units
   🔥 Sell at 30% Discount:     1186 units
   🗑️ Wasted / Spoiled:         0 units
--------------------------------------------------
📍 STORE: Koramangala
   🚚 Replenish from Warehouse: 600 units
   🍏 Sell at Full Price:       800 units
   🔥 Sell at 30% Discount:     0 units
   🗑️ Wasted / Spoiled:         0 units
--------------------------------------------------

🤖 SYSTEM REASONING:
Koramangala prioritized for replenishment due to high full-price demand and low spoilage risk.
Indiranagar prioritized for aggressive 30% discounting to liquidate 1,186 units before heatwave spoilage.
